In [2]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

In [3]:
df = pd.read_csv("../data/raw/Loan_Default.csv")

In [4]:
df.shape

(148670, 34)

In [5]:
df.dtypes

ID                             int64
year                           int64
loan_limit                    object
Gender                        object
approv_in_adv                 object
loan_type                     object
loan_purpose                  object
Credit_Worthiness             object
open_credit                   object
business_or_commercial        object
loan_amount                    int64
rate_of_interest             float64
Interest_rate_spread         float64
Upfront_charges              float64
term                         float64
Neg_ammortization             object
interest_only                 object
lump_sum_payment              object
property_value               float64
construction_type             object
occupancy_type                object
Secured_by                    object
total_units                   object
income                       float64
credit_type                   object
Credit_Score                   int64
co-applicant_credit_type      object
a

In [6]:
df['Status'].value_counts()

Status
0    112031
1     36639
Name: count, dtype: int64

##### here also dataset not balanced. clear 3:1 imbalance. so we cant use accuracy as a metric here. because if model predicts all 0, still it gives around 75% Accuracy.

so we have to pick metric that matches for this dataset

## Check for unique values to decide encoding methods

In [7]:
for col in df.select_dtypes(include=['object']).columns:
    unique_values = df[col].unique()
    print(f"\nColumn: {col}")
    print(f"Unique values ({len(unique_values)}):")
    print(unique_values)


Column: loan_limit
Unique values (3):
['cf' nan 'ncf']

Column: Gender
Unique values (4):
['Sex Not Available' 'Male' 'Joint' 'Female']

Column: approv_in_adv
Unique values (3):
['nopre' 'pre' nan]

Column: loan_type
Unique values (3):
['type1' 'type2' 'type3']

Column: loan_purpose
Unique values (5):
['p1' 'p4' 'p3' 'p2' nan]

Column: Credit_Worthiness
Unique values (2):
['l1' 'l2']

Column: open_credit
Unique values (2):
['nopc' 'opc']

Column: business_or_commercial
Unique values (2):
['nob/c' 'b/c']

Column: Neg_ammortization
Unique values (3):
['not_neg' 'neg_amm' nan]

Column: interest_only
Unique values (2):
['not_int' 'int_only']

Column: lump_sum_payment
Unique values (2):
['not_lpsm' 'lpsm']

Column: construction_type
Unique values (2):
['sb' 'mh']

Column: occupancy_type
Unique values (3):
['pr' 'sr' 'ir']

Column: Secured_by
Unique values (2):
['home' 'land']

Column: total_units
Unique values (4):
['1U' '2U' '3U' '4U']

Column: credit_type
Unique values (4):
['EXP' 'EQUI'

## Handle missing Values

In [8]:
df.isnull().sum()

ID                               0
year                             0
loan_limit                    3344
Gender                           0
approv_in_adv                  908
loan_type                        0
loan_purpose                   134
Credit_Worthiness                0
open_credit                      0
business_or_commercial           0
loan_amount                      0
rate_of_interest             36439
Interest_rate_spread         36639
Upfront_charges              39642
term                            41
Neg_ammortization              121
interest_only                    0
lump_sum_payment                 0
property_value               15098
construction_type                0
occupancy_type                   0
Secured_by                       0
total_units                      0
income                        9150
credit_type                      0
Credit_Score                     0
co-applicant_credit_type         0
age                            200
submission_of_applic

In [9]:
df.groupby('Status')['rate_of_interest'].apply(lambda x: x.isna().mean())

Status
0    0.000000
1    0.994541
Name: rate_of_interest, dtype: float64

 99.4% of all defaulted loans have rate_of_interest missing. That's not noise, that's a structural pattern. The missingness flag we're adding isn't just a nice-to-have, it's probably going to be one of the strongest predictive signals in the entire model. The model will essentially learn "if rate_of_interest is missing, predict default" — which is exactly what the data is telling us.

 So we cant drop `rate_of_interest` column

In [10]:
df.describe()

,ID,year,loan_amount,rate_of_interest,Interest_rate_spread,Upfront_charges,term,property_value,income,Credit_Score,LTV,Status,dtir1
count,148670.000000,148670.0,1.486700e+05,112231.000000,112031.000000,109028.000000,148629.000000,1.335720e+05,139520.000000,148670.000000,133572.000000,148670.000000,124549.000000
mean,99224.500000,2019.0,3.311177e+05,4.045476,0.441656,3224.996127,335.136582,4.978935e+05,6957.338876,699.789103,72.746457,0.246445,37.732932
std,42917.476598,0.0,1.839093e+05,0.561391,0.513043,3251.121510,58.409084,3.599353e+05,6496.586382,115.875857,39.967603,0.430942,10.545435
min,24890.000000,2019.0,1.650000e+04,0.000000,-3.638000,0.000000,96.000000,8.000000e+03,0.000000,500.000000,0.967478,0.000000,5.000000
25%,62057.250000,2019.0,1.965000e+05,3.625000,0.076000,581.490000,360.000000,2.680000e+05,3720.000000,599.000000,60.474860,0.000000,31.000000
50%,99224.500000,2019.0,2.965000e+05,3.990000,0.390400,2596.450000,360.000000,4.180000e+05,5760.000000,699.000000,75.135870,0.000000,39.000000
75%,136391.750000,2019.0,4.365000e+05,4.375000,0.775400,4812.500000,360.000000,6.280000e+05,8520.000000,800.000000,86.184211,0.000000,45.000000
max,173559.000000,2019.0,3.576500e+06,8.000000,3.357000,60000.000000,360.000000,1.650800e+07,578580.000000,900.000000,7831.250000,1.000000,61.000000


In [11]:
for col in df.columns:
    null_count = df[col].isnull().sum()

    # Only process columns with null values
    if null_count > 0:
        counts = df[col].value_counts(dropna=False)

        print(f"\n=== {col} ===")
        print(f"Null count: {null_count}")
        print(f"Unique values: {len(counts)}")

        if len(counts) > 5:
            print("Top 5 values by count:")
            print(counts.head(5))
        else:
            print("Values and counts:")
            print(counts)


=== loan_limit ===
Null count: 3344
Unique values: 3
Values and counts:
loan_limit
cf     135348
ncf      9978
NaN      3344
Name: count, dtype: int64

=== approv_in_adv ===
Null count: 908
Unique values: 3
Values and counts:
approv_in_adv
nopre    124621
pre       23141
NaN         908
Name: count, dtype: int64

=== loan_purpose ===
Null count: 134
Unique values: 5
Values and counts:
loan_purpose
p3     55934
p4     54799
p1     34529
p2      3274
NaN      134
Name: count, dtype: int64

=== rate_of_interest ===
Null count: 36439
Unique values: 132
Top 5 values by count:
rate_of_interest
NaN      36439
3.990    14455
3.625     8800
3.875     8592
3.750     8474
Name: count, dtype: int64

=== Interest_rate_spread ===
Null count: 36639
Unique values: 22517
Top 5 values by count:
Interest_rate_spread
 NaN      36639
-0.028       77
-0.038       64
-0.023       60
-0.173       56
Name: count, dtype: int64

=== Upfront_charges ===
Null count: 39642
Unique values: 58272
Top 5 values by coun

### loan_limit

In [12]:
df.groupby('Status')['loan_limit'].apply(lambda x: x.isna().mean())

Status
0    0.021985
1    0.024045
Name: loan_limit, dtype: float64

In [13]:
suspect_cols = ['loan_limit', 'approv_in_adv', 'Neg_ammortization', 'income', 'dtir1', 'age']
for col in suspect_cols:
    result = df.groupby('Status')[col].apply(lambda x: x.isna().mean())
    print(f"\n{col}:\n{result}")


loan_limit:
Status
0    0.021985
1    0.024045
Name: loan_limit, dtype: float64

approv_in_adv:
Status
0    0.005954
1    0.006578
Name: approv_in_adv, dtype: float64

Neg_ammortization:
Status
0    0.000794
1    0.000873
Name: Neg_ammortization, dtype: float64

income:
Status
0    0.070614
1    0.033816
Name: income, dtype: float64

dtir1:
Status
0    0.069722
1    0.445154
Name: dtir1, dtype: float64

age:
Status
0    0.000000
1    0.005459
Name: age, dtype: float64


### 

In [ ]:
df.groupby('Gender')['Status'].mean()